# 05 Agent Cockpit
Google Driveを永続ストレージとしてマウントし、RLDS Dataset Explorer、OpenVLA推論、Agent Trace用Gradio UIを起動します。

OpenVLA推論を使う場合は、`01_model_feasibility.ipynb`で構築した依存環境とCheckpointを利用してください。Dataset ExplorerとRule-based Plannerだけなら、このNotebook単体でも確認できます。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/yu37330/Physical_ai.git'
BRANCH = 'agent/add-colab-agent-cockpit'
!rm -rf /content/Physical_ai
!git clone --branch $BRANCH $REPO_URL /content/Physical_ai
%cd /content/Physical_ai
!pip install -q -r training/openvla_oft_a100/requirements-data.txt
!pip install -q -r requirements-frontend.txt

In [ ]:
import os
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/PARC2026')
os.environ['PHYSICAL_AI_DRIVE_ROOT'] = str(DRIVE_ROOT)

for relative in ['20_processed', '30_models', '40_experiments/agent_cockpit']:
    path = DRIVE_ROOT / relative
    path.mkdir(parents=True, exist_ok=True)
    print(relative, '->', path)

!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

UI起動後は、次の順序で確認します。

1. DashboardでRunを作成
2. Dataset ExplorerでManifestとTFDS Builder directoryを指定
3. RLDS Sampleを読み込み、画像・State・Target Action chunkを確認
4. InferenceでCheckpoint directoryを指定してOpenVLA推論
5. Agent Cockpitで`openvla` Plannerと`propose`モードを選び、次ActionをTraceへ保存

In [ ]:
from frontend.gradio_app import build_app
build_app().launch(share=True, debug=True)